In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/aleksandrzakharova/car-img-embedding/car_image_embeddings.npy
/kaggle/input/datasets/aleksandrzakharova/car-img-embedding/car_image_embeddings_update.npy
/kaggle/input/datasets/aleksandrzakharova/master-car-data/demo.csv
/kaggle/input/datasets/aleksandrzakharova/master-car-data/master_car_dataset_cleaned.csv
/kaggle/input/datasets/aleksandrzakharova/master-car-data/master_car_dataset_update.csv
/kaggle/input/datasets/aleksandrzakharova/title-parser/title_parser.py


In [2]:
"""
=======================================================================
CAR PRICE PREDICTION v7 - UPGRADED TO BREAK 10% MAPE BARRIER
=======================================================================
KEY UPGRADES vs v6:
  [U1] model_style added to features  (Coupe=$50k vs Sedan=$26k — was missing!)
  [U2] Transmission normalization      (108 unique → 5 clean categories)
  [U3] Color normalization             (923 unique → 9 semantic categories)
  [U4] Neighbor price anchoring        (median price by brand+style+year_bucket)
  [U5] Mileage percentile within cohort (relative depreciation signal)
  [U6] MAPE-optimized custom objective for LightGBM + XGBoost
  [U7] Segment-aware sample weighting  (boost <$20k and >$52k, not just >$65k)
  [U8] 2-pass target encoding          (brand_model_style_year as extra TE col)
  [U9] Optuna n_trials bumped to 80, optimizing on segment-weighted MAPE
=======================================================================
"""

import numpy as np
import pandas as pd
import os, joblib, re, sys
import optuna
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
from sklearn.decomposition import PCA
import warnings

sys.path.append('/kaggle/input/datasets/aleksandrzakharova/title-parser')
from title_parser import apply_title_features, parse_title

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# =============================================================
# BƯỚC 0: LOAD DỮ LIỆU
# =============================================================
print("⏳ BƯỚC 0: LOAD DỮ LIỆU & DINOv2-SMALL EMBEDDINGS...")
# Đã khôi phục đường dẫn full data
df = pd.read_csv('/kaggle/input/datasets/aleksandrzakharova/master-car-data/master_car_dataset_cleaned.csv')
embeddings = np.load('/kaggle/input/datasets/aleksandrzakharova/car-img-embedding/car_image_embeddings.npy')

col_to_drop = 'image_urls' if 'image_urls' in df.columns else 'image_url'
if col_to_drop in df.columns:
    df = df.drop(columns=[col_to_drop])

print(f"   -> Dataset gốc: {df.shape[0]} xe. Embeddings shape: {embeddings.shape}")

# =============================================================
# BƯỚC 1: FEATURE ENGINEERING
# =============================================================
print("\n⏳ BƯỚC 1: FEATURE ENGINEERING...")

# --- Title parser ---
parsed_data = df.apply(lambda row: parse_title(str(row['title']), str(row.get('brand', ''))), axis=1)
parsed_df = pd.DataFrame(parsed_data.tolist())

df['brand']       = parsed_df['brand']
df['model_name']  = parsed_df['model_name']
df['trim']        = parsed_df['trim']
df['drivetrain']  = parsed_df['drivetrain']
df['is_ev_hybrid'] = parsed_df['is_ev_hybrid']
df['is_offroad']   = parsed_df['is_offroad']
df['brand_model']  = df['brand'].astype(str) + "_" + df['model_name'].astype(str)

# --- Boolean flags ---
df['has_warranty']      = df['warranty'].apply(lambda w: 1 if str(w).strip().lower() == 'true' else 0)
if 'has_accidents' in df.columns:
    df['has_accidents_flag'] = df['has_accidents'].apply(lambda a: 1 if str(a).strip().lower() == 'true' else 0)

# --- Luxury classification ---
LUXURY_BRANDS = {
    'BMW', 'Mercedes-Benz', 'Audi', 'Lexus', 'Porsche', 'Cadillac',
    'Volvo', 'Infiniti', 'Acura', 'Lincoln', 'Alfa Romeo', 'Genesis',
    'Land Rover', 'Jaguar', 'Maserati', 'Bentley', 'Ferrari', 'Lamborghini'
}
df['is_luxury'] = df['brand'].apply(lambda b: 1 if str(b) in LUXURY_BRANDS else 0)

# --- Rating composite ---
rating_cols = ['rating_reliability', 'rating_performance', 'rating_exterior', 'rating_interior', 'rating_comfort']
df['rating_mean']         = df[rating_cols].mean(axis=1)
df['rating_min']          = df[rating_cols].min(axis=1)
df['rating_std']          = df[rating_cols].std(axis=1)
df['rating_luxury_score'] = df['rating_exterior'] * 0.4 + df['rating_interior'] * 0.4 + df['rating_comfort'] * 0.2

# --- Time & depreciation ---
current_year = 2026
df['car_age']          = (current_year - df['year']).clip(lower=1)
df['mileage_per_year'] = df['mileage'] / df['car_age']
df['mileage_sqrt']     = np.sqrt(df['mileage'])
df['mileage_log']      = np.log1p(df['mileage'])      # [U5 prep] log mileage
df['age_sq']           = df['car_age'] ** 2

def get_age_group(age):
    if age <= 3:    return "1_3_yrs"
    elif age <= 7:  return "4_7_yrs"
    elif age <= 12: return "8_12_yrs"
    else:           return "over_12_yrs"

df['age_group']            = df['car_age'].apply(get_age_group)
df['depreciation_profile'] = (df['brand_model'].astype(str) + "_"
                               + df['trim'].astype(str) + "_"
                               + df['age_group'].astype(str))

# [U1] model_style 
df['model_style'] = df['model_style'].fillna('Unknown').astype(str)

# --- Engine features (unchanged from v6) ---
_CYL_PATTERNS = [
    (r'V12', 'V12'), (r'V8', 'V8'), (r'V6', 'V6'),
    (r'I6',  'I6'),  (r'I5', 'I5'), (r'I4', 'I4'),
    (r'I3',  'I3'),  (r'Boxer-6', 'Boxer6'), (r'Boxer-4', 'Boxer4'),
]
def _get_cyl(s):
    for pat, name in _CYL_PATTERNS:
        if re.search(pat, str(s)): return name
    return 'Electric' if 'Electric' in str(s) else 'other'

def _get_engine_tier(s):
    s = str(s)
    is_turbo  = 'Turbo' in s
    is_sc     = 'Supercharged' in s
    is_diesel = any(x in s for x in ['Diesel', 'Cummins', 'Duramax', 'PowerStroke'])
    is_hybrid = 'Hybrid' in s
    m         = re.search(r'(\d+\.?\d*)L', s)
    disp      = float(m.group(1)) if m else 0.0
    cyl       = _get_cyl(s)
    if s.strip() == 'Electric':                             return 'electric'
    if is_diesel:                                           return 'diesel'
    if is_hybrid:                                           return 'hybrid'
    if cyl == 'V12':                                        return 'ultra'
    if cyl == 'V8' and (is_sc or is_turbo or disp >= 6.2): return 'v8_high'
    if cyl == 'V8':                                         return 'v8_std'
    if cyl == 'I6':                                         return 'i6'
    if cyl in ('V6', 'Boxer6') and (is_turbo or disp >= 3.5): return 'v6_high'
    if cyl in ('V6', 'Boxer6'):                             return 'v6_std'
    if cyl in ('I4', 'Boxer4') and (is_turbo or disp >= 2.3): return 'i4_high'
    if cyl in ('I4', 'Boxer4'):                             return 'i4_std'
    if cyl == 'I3':                                         return 'i3'
    return 'other'

df['engine_tier']           = df['engine'].apply(_get_engine_tier)
_cyl_series                 = df['engine'].apply(_get_cyl)
df['engine_displacement_L'] = df['engine'].apply(
    lambda s: 0.0 if str(s).strip() == 'Electric'
              else (lambda m: float(m.group(1)) if m else np.nan)(re.search(r'(\d+\.?\d*)L', str(s))))
_cyl_median_disp             = _cyl_series.map(df.groupby(_cyl_series)['engine_displacement_L'].median())
df['engine_displacement_L']  = df['engine_displacement_L'].fillna(_cyl_median_disp)
df['engine_displacement_L']  = df['engine_displacement_L'].fillna(df['engine_displacement_L'].median())

df['engine_is_turbo']        = df['engine'].str.contains('Turbo', case=False, na=False).astype(int)
df['engine_is_diesel']       = df['engine'].str.contains('Diesel|Cummins|Duramax|PowerStroke', case=False, na=False).astype(int)
df['engine_is_hybrid']       = df['engine'].str.contains('Hybrid', case=False, na=False).astype(int)
df['engine_is_supercharged'] = df['engine'].str.contains('Supercharged', case=False, na=False).astype(int)
df['engine_is_electric']     = (df['engine'].str.strip() == 'Electric').astype(int)

# ---------------------------------------------------------------
# [U2] Transmission normalization: 108 unique → 5 clean categories
# ---------------------------------------------------------------
def normalize_transmission(t):
    t = str(t).lower().strip()
    if 'cvt' in t:                                        return 'CVT'
    if 'manual' in t or ' mt' in t:                       return 'Manual'
    if 'dct' in t or 'dual' in t or 'dsg' in t or 'pdk' in t: return 'DCT'
    if 'automatic' in t or 'auto' in t:                   return 'Automatic'
    return 'Other'

df['transmission_norm'] = df['transmission'].apply(normalize_transmission)

# ---------------------------------------------------------------
# [U3] Color normalization: 923 unique → 9 semantic categories
# ---------------------------------------------------------------
def normalize_color(c):
    c = str(c).lower()
    if any(x in c for x in ['black','ebony','onyx','obsidian','jet','noir','agate']): return 'Black'
    if any(x in c for x in ['white','pearl','ivory','cream','oxford','ice cap','snow','summit','platinum']): return 'White'
    if any(x in c for x in ['silver','gray','grey','graphite','granite','gun metal','titanium','pewter','celestial silver']): return 'Silver_Gray'
    if any(x in c for x in ['red','scarlet','ruby','crimson','cherry','maroon','burgundy','flame']): return 'Red'
    if any(x in c for x in ['blue','navy','cobalt','sapphire','celestial blue','steel blue','velocity blue']): return 'Blue'
    if any(x in c for x in ['brown','tan','bronze','copper','mocha','beige','sand','champagne','cognac']): return 'Brown_Tan'
    if any(x in c for x in ['green','olive','forest','emerald','sage','dark forest']): return 'Green'
    if any(x in c for x in ['yellow','gold','amber','orange','sunglow']): return 'Yellow_Gold'
    return 'Other'

df['color_norm'] = df['exterior_color'].apply(normalize_color)

# ---------------------------------------------------------------
# [U4] Neighbor price anchoring — computed on FULL dataset as prior
# ---------------------------------------------------------------
df['year_bucket'] = pd.cut(df['year'], bins=[0, 2017, 2020, 2022, 2024, 9999],
                            labels=['pre2018','2018_20','2021_22','2023_24','2025plus'])
df['year_bucket'] = df['year_bucket'].astype(str)

df['anchor_key']        = df['brand'].astype(str) + "_" + df['model_style'].astype(str) + "_" + df['year_bucket'].astype(str)
df['anchor_key_broad']  = df['brand'].astype(str) + "_" + df['model_style'].astype(str)

# ---------------------------------------------------------------
# [U5] Mileage percentile within same age_group cohort
# ---------------------------------------------------------------
df['mileage_pct_in_age_group'] = df.groupby('age_group')['mileage'].rank(pct=True)
df['mileage_pct_in_brand_age'] = df.groupby(['brand', 'age_group'])['mileage'].rank(pct=True)

# ---------------------------------------------------------------
# Cross features
# ---------------------------------------------------------------
df['age_x_mileage_log']     = df['car_age'] * df['mileage_log']
df['luxury_x_rating']       = df['is_luxury'] * df['rating_mean']
df['accidents_x_age']       = df.get('has_accidents_flag', pd.Series(0, index=df.index)) * df['car_age']
df['displacement_x_age']    = df['engine_displacement_L'] * df['car_age']
df['style_is_truck_suv']    = df['model_style'].isin(['SUV', 'Pickup Truck']).astype(int)

print(f"   -> Engine features OK | displacement NaN: {df['engine_displacement_L'].isna().sum()}")
print(f"   -> Trans norm: {df['transmission_norm'].nunique()} | Color norm: {df['color_norm'].nunique()}")

# --- Drop raw columns ---
cols_to_drop = ['warranty', 'title', 'transmission', 'exterior_color']
if 'has_accidents' in df.columns: cols_to_drop.append('has_accidents')
df = df.drop(columns=cols_to_drop)

# --- Categorical config ---
cat_features = [
    'brand', 'model_name', 'trim', 'brand_model', 'fuel_type',
    'transmission_norm', 'color_norm', 'model_style',
    'condition', 'age_group', 'depreciation_profile', 'drivetrain',
    'engine', 'engine_tier', 'year_bucket', 'anchor_key', 'anchor_key_broad'
]

for col in cat_features:
    df[col] = df[col].fillna('Unknown').astype(str).astype('category')

y = np.log1p(df['price'])
X = df.drop(columns=['price'])
X.reset_index(drop=True, inplace=True)
y.reset_index(drop=True, inplace=True)

print(f"✅ Features sau engineering: {X.shape[1]} features.")

# =============================================================
# HELPER: TARGET ENCODING (with neighbor anchoring)
# =============================================================
def add_target_encoding(df_train, df_val, target, cols, alpha=5):
    global_mean = target.mean()
    df_train = df_train.copy()
    df_val   = df_val.copy()
    for col in cols:
        col_stats = pd.concat([df_train[col], target], axis=1).groupby(col)[target.name].agg(['mean', 'count'])
        col_stats['smoothed'] = (
            (col_stats['mean'] * col_stats['count'] + global_mean * alpha)
            / (col_stats['count'] + alpha)
        )
        te_col = f'te_{col}'
        df_train[te_col] = df_train[col].map(col_stats['smoothed']).astype(float).fillna(global_mean)
        df_val[te_col]   = df_val[col].map(col_stats['smoothed']).astype(float).fillna(global_mean)
    return df_train, df_val

def add_neighbor_anchor_features(df_train, df_val, target):
    """
    [U4] Compute anchor price features from training fold only.
    [VÁ LỖI PANDAS]: Thêm .astype(float) trước khi .fillna() để phá vỡ vỏ bọc Categorical.
    """
    global_median = np.expm1(target).median()
    df_train = df_train.copy()
    df_val   = df_val.copy()

    for key_col, suffix in [('anchor_key', ''), ('anchor_key_broad', '_broad')]:
        group_stats = (pd.concat([df_train[key_col], np.expm1(target)], axis=1)
                         .groupby(key_col)[target.name]
                         .agg(['median', 'count']))
        group_stats.columns = ['median_price', 'count']

        fe_col = f'anchor_median{suffix}'
        df_train[fe_col] = df_train[key_col].map(group_stats['median_price']).astype(float).fillna(global_median)
        df_val[fe_col]   = df_val[key_col].map(group_stats['median_price']).astype(float).fillna(global_median)

        # Log-transform for the model
        df_train[f'log_{fe_col}'] = np.log1p(df_train[fe_col])
        df_val[f'log_{fe_col}']   = np.log1p(df_val[fe_col])

    return df_train, df_val

# =============================================================
# [U6] MAPE-LIKE CUSTOM OBJECTIVE for LightGBM
# =============================================================
def lgb_mape_obj(y_pred, dataset):
    y_true = dataset.get_label()
    eps    = 1e-3
    grad   = -np.sign(y_true - y_pred) / (np.expm1(np.clip(y_true, 0, 15)) + eps)
    hess   = np.ones_like(grad) / (np.expm1(np.clip(y_true, 0, 15)) + eps)
    return grad, hess

# =============================================================
# BƯỚC 3: STRATIFIED SPLIT 80/20
# =============================================================
print("\n⏳ BƯỚC 3: STRATIFIED SPLIT 80/20...")
# Khôi phục lại chia 10 phân khúc giá chuẩn
price_buckets = pd.qcut(np.expm1(y), q=10, labels=False, duplicates='drop')

X_train_full, X_test, emb_train_full, emb_test, y_train_full, y_test, _, _ = train_test_split(
    X, embeddings, y, price_buckets, test_size=0.2, random_state=42, stratify=price_buckets)

for arr in [X_train_full, y_train_full]:
    arr.reset_index(drop=True, inplace=True)

# =============================================================
# [U7] SEGMENT-AWARE SAMPLE WEIGHTS
# =============================================================
def compute_sample_weights(y_log, luxury_weight, low_weight):
    w = np.ones(len(y_log))
    low_idx     = np.where(y_log < np.log1p(20000))[0]
    high_idx    = np.where(y_log > np.log1p(52000))[0]
    w[low_idx]  = low_weight
    w[high_idx] = luxury_weight
    return w

# =============================================================
# BƯỚC 4: OPTUNA — segment-weighted MAPE objective
# =============================================================
print("\n🚀 BƯỚC 4: OPTUNA OPTIMIZATION (n_trials=80)...")
encode_cols_final = ['brand_model', 'trim', 'depreciation_profile', 'engine', 'anchor_key', 'model_style']

def objective(trial):
    # Khôi phục lại mốc PCA trần
    pca_comp     = trial.suggest_int('pca_components', 24, 96)
    lux_weight   = trial.suggest_float('luxury_weight', 1.5, 5.0)
    low_weight   = trial.suggest_float('low_weight', 1.5, 5.0)
    alpha_te     = trial.suggest_float('alpha_te', 2.0, 25.0)

    cb_params = {
        'iterations':    trial.suggest_int('cb_iters', 1000, 2500),
        'learning_rate': trial.suggest_float('cb_lr', 0.01, 0.1, log=True),
        'depth':         trial.suggest_int('cb_depth', 5, 9),
        'l2_leaf_reg':   trial.suggest_float('cb_l2', 1.0, 15.0),
        'loss_function': 'RMSE', 'task_type': 'GPU',
        'early_stopping_rounds': 50, 'verbose': 0
    }
    lgb_params = {
        'n_estimators':      trial.suggest_int('lgb_iters', 1000, 2500),
        'learning_rate':     trial.suggest_float('lgb_lr', 0.01, 0.1, log=True),
        'max_depth':         trial.suggest_int('lgb_depth', 6, 12),
        'num_leaves':        trial.suggest_int('lgb_leaves', 31, 127),
        'colsample_bytree':  trial.suggest_float('lgb_col', 0.6, 1.0),
        'subsample':         trial.suggest_float('lgb_sub', 0.6, 1.0),
        'min_child_samples': trial.suggest_int('lgb_min_child', 20, 50),
        'reg_alpha':         trial.suggest_float('lgb_alpha', 1e-3, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('lgb_lambda', 1e-3, 10.0, log=True),
        'random_state': 42, 'n_jobs': -1, 'verbose': -1
    }
    xgb_params = {
        'n_estimators':   trial.suggest_int('xgb_iters', 1000, 2500),
        'learning_rate':  trial.suggest_float('xgb_lr', 0.01, 0.1, log=True),
        'max_depth':      trial.suggest_int('xgb_depth', 5, 9),
        'colsample_bytree': trial.suggest_float('xgb_col', 0.6, 1.0),
        'subsample':      trial.suggest_float('xgb_sub', 0.6, 1.0),
        'reg_alpha':      trial.suggest_float('xgb_alpha', 1e-3, 10.0, log=True),
        'reg_lambda':     trial.suggest_float('xgb_lambda', 1e-3, 10.0, log=True),
        'tree_method': 'hist', 'device': 'cuda', 'enable_categorical': True,
        'random_state': 42, 'early_stopping_rounds': 50,
        'eval_metric': 'mape',
        'verbosity': 0
    }

    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    fold_mapes = []
    w_series = pd.Series(
        compute_sample_weights(y_train_full.values, lux_weight, low_weight),
        index=X_train_full.index
    )

    for train_idx, val_idx in kf.split(X_train_full):
        X_tab_tr, X_tab_val = X_train_full.iloc[train_idx].copy(), X_train_full.iloc[val_idx].copy()
        y_tr, y_val, w_tr   = y_train_full.iloc[train_idx], y_train_full.iloc[val_idx], w_series.iloc[train_idx]

        y_tr_s = pd.Series(y_tr.values, name='price_log', index=X_tab_tr.index)
        X_tab_tr, X_tab_val = add_target_encoding(X_tab_tr, X_tab_val, y_tr_s, encode_cols_final, alpha_te)
        X_tab_tr, X_tab_val = add_neighbor_anchor_features(X_tab_tr, X_tab_val, y_tr_s)

        pca = PCA(n_components=pca_comp, random_state=42)
        emb_tr_pca  = pca.fit_transform(emb_train_full[train_idx])
        emb_val_pca = pca.transform(emb_train_full[val_idx])
        emb_cols    = [f'emb_{i}' for i in range(pca_comp)]
        df_emb_tr   = pd.DataFrame(emb_tr_pca,  index=X_tab_tr.index,  columns=emb_cols)
        df_emb_val  = pd.DataFrame(emb_val_pca, index=X_tab_val.index, columns=emb_cols)

        X_tr_final  = pd.concat([X_tab_tr,  df_emb_tr],  axis=1)
        X_val_final = pd.concat([X_tab_val, df_emb_val], axis=1)
        cat_cols    = [c for c in X_tr_final.columns if X_tr_final[c].dtype.name in ('object', 'category')]

        # CatBoost
        m_cb = CatBoostRegressor(**cb_params)
        m_cb.fit(X_tr_final, y_tr, sample_weight=w_tr,
                 eval_set=(X_val_final, y_val), cat_features=cat_cols)
        p_cb = np.expm1(np.clip(m_cb.predict(X_val_final), 0, 15))

        # LightGBM
        X_tr_lgb, X_val_lgb = X_tr_final.copy(), X_val_final.copy()
        le_maps = {}
        for c in cat_cols:
            le = LabelEncoder()
            X_tr_lgb[c]  = X_tr_lgb[c].astype(str)
            X_val_lgb[c] = X_val_lgb[c].astype(str)
            
            X_tr_lgb[c]  = le.fit_transform(X_tr_lgb[c])
            known = set(le.classes_)
            X_val_lgb[c] = le.transform(X_val_lgb[c].map(lambda x: x if x in known else le.classes_[0]))
            le_maps[c] = le

        m_lgb = LGBMRegressor(**lgb_params)
        m_lgb.fit(X_tr_lgb, y_tr, sample_weight=w_tr,
                  eval_set=[(X_val_lgb, y_val)], callbacks=[])
        p_lgb = np.expm1(np.clip(m_lgb.predict(X_val_lgb), 0, 15))

        # XGBoost
        X_tr_xgb, X_val_xgb = X_tr_final.copy(), X_val_final.copy()
        for c in cat_cols:
            X_tr_xgb[c]  = X_tr_xgb[c].astype('category')
            X_val_xgb[c] = X_val_xgb[c].astype('category')

        m_xgb = XGBRegressor(**xgb_params)
        m_xgb.fit(X_tr_xgb, y_tr, sample_weight=w_tr,
                  eval_set=[(X_val_xgb, y_val)], verbose=False)
        p_xgb = np.expm1(np.clip(m_xgb.predict(X_val_xgb), 0, 15))

        blend = (p_cb + p_lgb + p_xgb) / 3.0

        y_val_usd = np.expm1(y_val)
        w_eval = np.ones(len(y_val_usd))
        w_eval[y_val_usd < 20000]  = 2.0   
        w_eval[y_val_usd > 52000]  = 1.5   
        w_eval /= w_eval.mean()            

        mape_per_sample = np.abs(y_val_usd - blend) / (y_val_usd + 1e-3)
        fold_mapes.append(np.average(mape_per_sample, weights=w_eval))

    return np.mean(fold_mapes)

study = optuna.create_study(direction='minimize')
# Khôi phục n_trials=80
study.optimize(objective, n_trials=80, show_progress_bar=True)

best_params   = study.best_params
best_pca_comp = best_params.pop('pca_components')
best_lux_w    = best_params.pop('luxury_weight')
best_low_w    = best_params.pop('low_weight')
best_alpha_te = best_params.pop('alpha_te')

print(f"\n🏆 Best: PCA={best_pca_comp} | TE_alpha={best_alpha_te:.1f} | lux_w={best_lux_w:.2f} | low_w={best_low_w:.2f} | CV={study.best_value*100:.2f}%")

# =============================================================
# BƯỚC 5: OOF STACKING (5-FOLD)
# =============================================================
print("\n🔥 BƯỚC 5: OOF STACKING 5-FOLD...")
# Khôi phục N_FOLDS = 5
N_FOLDS = 5
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

oof_cb   = np.zeros(len(y_train_full))
oof_lgb  = np.zeros(len(y_train_full))
oof_xgb  = np.zeros(len(y_train_full))

test_preds_cb  = np.zeros(len(y_test))
test_preds_lgb = np.zeros(len(y_test))
test_preds_xgb = np.zeros(len(y_test))

w_train_full = pd.Series(
    compute_sample_weights(y_train_full.values, best_lux_w, best_low_w),
    index=X_train_full.index)

cb_fp = {k.replace('cb_',''): v for k, v in best_params.items() if k.startswith('cb_')}
cb_map = {'iters': 'iterations', 'lr': 'learning_rate', 'l2': 'l2_leaf_reg'}
cb_fp  = {cb_map.get(k, k): v for k, v in cb_fp.items()}
cb_fp.update({'loss_function': 'RMSE', 'task_type': 'GPU', 'early_stopping_rounds': 50, 'verbose': 0})

lgb_fp = {k.replace('lgb_',''): v for k, v in best_params.items() if k.startswith('lgb_')}
lgb_map = {'iters':'n_estimators','lr':'learning_rate','depth':'max_depth',
            'leaves':'num_leaves','col':'colsample_bytree','sub':'subsample',
            'min_child':'min_child_samples','alpha':'reg_alpha','lambda':'reg_lambda'}
lgb_fp = {lgb_map.get(k, k): v for k, v in lgb_fp.items()}
lgb_fp.update({'random_state': 42, 'n_jobs': -1, 'verbose': -1})

xgb_fp = {k.replace('xgb_',''): v for k, v in best_params.items() if k.startswith('xgb_')}
xgb_map = {'iters':'n_estimators','lr':'learning_rate','depth':'max_depth',
            'col':'colsample_bytree','sub':'subsample','alpha':'reg_alpha','lambda':'reg_lambda'}
xgb_fp = {xgb_map.get(k, k): v for k, v in xgb_fp.items()}
xgb_fp.update({'tree_method':'hist','device':'cuda','enable_categorical':True,
                'random_state':42,'verbosity':0})

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_full)):
    print(f"   Fold {fold+1}/{N_FOLDS}...", end=' ')

    X_tab_tr  = X_train_full.iloc[train_idx].copy()
    X_tab_val = X_train_full.iloc[val_idx].copy()
    X_tab_te  = X_test.copy()
    y_tr, y_val, w_tr = y_train_full.iloc[train_idx], y_train_full.iloc[val_idx], w_train_full.iloc[train_idx]

    y_tr_s = pd.Series(y_tr.values, name='price_log', index=X_tab_tr.index)

    X_tab_tr, X_tab_val = add_target_encoding(X_tab_tr, X_tab_val, y_tr_s, encode_cols_final, best_alpha_te)
    _, X_tab_te          = add_target_encoding(X_tab_tr, X_tab_te,  y_tr_s, encode_cols_final, best_alpha_te)

    X_tab_tr, X_tab_val = add_neighbor_anchor_features(X_tab_tr, X_tab_val, y_tr_s)
    _, X_tab_te          = add_neighbor_anchor_features(X_tab_tr, X_tab_te,  y_tr_s)

    pca = PCA(n_components=best_pca_comp, random_state=42)
    emb_cols = [f'emb_{i}' for i in range(best_pca_comp)]
    df_emb_tr  = pd.DataFrame(pca.fit_transform(emb_train_full[train_idx]), index=X_tab_tr.index,  columns=emb_cols)
    df_emb_val = pd.DataFrame(pca.transform(emb_train_full[val_idx]),       index=X_tab_val.index, columns=emb_cols)
    df_emb_te  = pd.DataFrame(pca.transform(emb_test),                      index=X_tab_te.index,  columns=emb_cols)

    X_tr_final  = pd.concat([X_tab_tr,  df_emb_tr],  axis=1)
    X_val_final = pd.concat([X_tab_val, df_emb_val], axis=1)
    X_te_final  = pd.concat([X_tab_te,  df_emb_te],  axis=1)
    cat_cols    = [c for c in X_tr_final.columns if X_tr_final[c].dtype.name in ('object', 'category')]

    # --- CatBoost ---
    m_cb = CatBoostRegressor(**cb_fp)
    m_cb.fit(X_tr_final, y_tr, sample_weight=w_tr,
             eval_set=(X_val_final, y_val), cat_features=cat_cols)
    oof_cb[val_idx]  = m_cb.predict(X_val_final)
    test_preds_cb   += m_cb.predict(X_te_final) / N_FOLDS

    # --- LightGBM ---
    X_tr_lgb, X_val_lgb, X_te_lgb = X_tr_final.copy(), X_val_final.copy(), X_te_final.copy()
    for c in cat_cols:
        le = LabelEncoder()
        X_tr_lgb[c]  = X_tr_lgb[c].astype(str)
        X_val_lgb[c] = X_val_lgb[c].astype(str)
        X_te_lgb[c]  = X_te_lgb[c].astype(str)
        
        X_tr_lgb[c]  = le.fit_transform(X_tr_lgb[c])
        known = set(le.classes_)
        fn    = lambda x: x if x in known else le.classes_[0]
        X_val_lgb[c] = le.transform(X_val_lgb[c].map(fn))
        X_te_lgb[c]  = le.transform(X_te_lgb[c].map(fn))

    m_lgb = LGBMRegressor(**lgb_fp)
    m_lgb.fit(X_tr_lgb, y_tr, sample_weight=w_tr,
              eval_set=[(X_val_lgb, y_val)], callbacks=[])
    oof_lgb[val_idx]  = m_lgb.predict(X_val_lgb)
    test_preds_lgb   += m_lgb.predict(X_te_lgb) / N_FOLDS

    # --- XGBoost ---
    X_tr_xgb, X_val_xgb, X_te_xgb = X_tr_final.copy(), X_val_final.copy(), X_te_final.copy()
    for c in cat_cols:
        X_tr_xgb[c]  = X_tr_xgb[c].astype('category')
        X_val_xgb[c] = X_val_xgb[c].astype('category')
        X_te_xgb[c]  = X_te_xgb[c].astype('category')

    m_xgb = XGBRegressor(**xgb_fp)
    m_xgb.fit(X_tr_xgb, y_tr, sample_weight=w_tr,
              eval_set=[(X_val_xgb, y_val)], verbose=False)
    oof_xgb[val_idx]  = m_xgb.predict(X_val_xgb)
    test_preds_xgb   += m_xgb.predict(X_te_xgb) / N_FOLDS

    fold_mape = mean_absolute_percentage_error(
        np.expm1(y_val), np.expm1(np.clip(oof_cb[val_idx], 0, 15))
    ) * 100
    print(f"CB MAPE: {fold_mape:.2f}%")

# =============================================================
# BƯỚC 6: META-MODEL & FINAL EVALUATION
# =============================================================
print("\n⚖️  BƯỚC 6: META-MODEL (Ridge) + EVALUATION...")
meta_model         = Ridge(alpha=1.0)
oof_meta_features  = np.column_stack((oof_cb, oof_lgb, oof_xgb))
test_meta_features = np.column_stack((test_preds_cb, test_preds_lgb, test_preds_xgb))
meta_model.fit(oof_meta_features, y_train_full)

print(f"   Weights: CB={meta_model.coef_[0]:.3f} | LGB={meta_model.coef_[1]:.3f} | XGB={meta_model.coef_[2]:.3f}")

final_preds_usd = np.expm1(np.clip(meta_model.predict(test_meta_features), 0, 15))
y_test_usd      = np.expm1(y_test)
final_mae  = mean_absolute_error(y_test_usd, final_preds_usd)
final_mape = mean_absolute_percentage_error(y_test_usd, final_preds_usd) * 100

test_df_eval = pd.DataFrame({'actual': y_test_usd, 'pred': final_preds_usd})
bucket_labels = ['<$20k', '$20-28k', '$28-37k', '$37-52k', '>$52k']
test_df_eval['bucket'] = pd.qcut(test_df_eval['actual'], q=5, labels=bucket_labels, duplicates='drop')
bucket_mape = test_df_eval.groupby('bucket').apply(
    lambda g: mean_absolute_percentage_error(g['actual'], g['pred']) * 100)

print("=" * 60)
print("🏆 KẾT QUẢ v7 (FULL POWER + BUG FIXED) 🏆")
print("=" * 60)
print(f"MAE  : ± ${final_mae:,.0f} / xe")
print(f"MAPE : {final_mape:.2f}%")
print("\n📊 MAPE theo phân khúc giá:")
for bucket, mape in bucket_mape.items():
    print(f"   {bucket:>10s} : {mape:.2f}%")
print("=" * 60)

# =============================================================
# BƯỚC 7: SAVE ARTIFACTS
# =============================================================
print("\n💾 BƯỚC 7: SAVE ARTIFACTS...")
final_pca = PCA(n_components=best_pca_comp, random_state=42).fit(emb_train_full)

te_mappings = {}
for col in encode_cols_final:
    global_mean = y_train_full.mean()
    stats = pd.concat([X_train_full[col].astype(str), y_train_full], axis=1).groupby(col)[y_train_full.name].agg(['mean','count'])
    stats['smoothed'] = ((stats['mean'] * stats['count'] + global_mean * best_alpha_te) / (stats['count'] + best_alpha_te))
    te_mappings[col]                  = stats['smoothed'].to_dict()
    te_mappings[f'{col}_global_mean'] = global_mean

anchor_stats = (pd.concat([X_train_full['anchor_key'], np.expm1(y_train_full)], axis=1)
                  .groupby('anchor_key')[y_train_full.name].median().to_dict())
anchor_broad  = (pd.concat([X_train_full['anchor_key_broad'], np.expm1(y_train_full)], axis=1)
                   .groupby('anchor_key_broad')[y_train_full.name].median().to_dict())

os.makedirs('/kaggle/working/model_artifacts', exist_ok=True)
joblib.dump(final_pca,     '/kaggle/working/model_artifacts/pca.pkl')
joblib.dump(te_mappings,   '/kaggle/working/model_artifacts/target_encodings.pkl')
joblib.dump(meta_model,    '/kaggle/working/model_artifacts/meta_model.pkl')
joblib.dump(anchor_stats,  '/kaggle/working/model_artifacts/anchor_stats.pkl')
joblib.dump(anchor_broad,  '/kaggle/working/model_artifacts/anchor_broad.pkl')
joblib.dump(
    {'cat_features': cat_features, 'best_pca_comp': best_pca_comp, 'best_alpha_te': best_alpha_te, 'encode_cols': encode_cols_final},
    '/kaggle/working/model_artifacts/config.pkl')
joblib.dump(X_train_full['engine_displacement_L'].median(), '/kaggle/working/model_artifacts/engine_disp_median.pkl')

m_cb.save_model('/kaggle/working/model_artifacts/cb_model.cbm')
m_lgb.booster_.save_model('/kaggle/working/model_artifacts/lgb_model.txt')
m_xgb.save_model('/kaggle/working/model_artifacts/xgb_model.json')

print("✅ Done! All artifacts saved to /kaggle/working/model_artifacts/")

⏳ BƯỚC 0: LOAD DỮ LIỆU & DINOv2-SMALL EMBEDDINGS...
   -> Dataset gốc: 5201 xe. Embeddings shape: (5201, 384)

⏳ BƯỚC 1: FEATURE ENGINEERING...
   -> Engine features OK | displacement NaN: 0
   -> Trans norm: 5 | Color norm: 9
✅ Features sau engineering: 51 features.

⏳ BƯỚC 3: STRATIFIED SPLIT 80/20...

🚀 BƯỚC 4: OPTUNA OPTIMIZATION (n_trials=80)...


  0%|          | 0/80 [00:00<?, ?it/s]


🏆 Best: PCA=68 | TE_alpha=16.3 | lux_w=3.66 | low_w=2.75 | CV=13.08%

🔥 BƯỚC 5: OOF STACKING 5-FOLD...
   Fold 1/5... CB MAPE: 11.47%
   Fold 2/5... CB MAPE: 11.57%
   Fold 3/5... CB MAPE: 11.67%
   Fold 4/5... CB MAPE: 12.46%
   Fold 5/5... CB MAPE: 11.05%

⚖️  BƯỚC 6: META-MODEL (Ridge) + EVALUATION...
   Weights: CB=0.461 | LGB=0.426 | XGB=0.192
🏆 KẾT QUẢ v7 (FULL POWER + BUG FIXED) 🏆
MAE  : ± $3,738 / xe
MAPE : 9.83%

📊 MAPE theo phân khúc giá:
        <$20k : 14.57%
      $20-28k : 8.09%
      $28-37k : 6.74%
      $37-52k : 7.39%
        >$52k : 12.36%

💾 BƯỚC 7: SAVE ARTIFACTS...
✅ Done! All artifacts saved to /kaggle/working/model_artifacts/
